# Project: Advanced Credit Risk Prediction
## Notebook 01: Data Acquisition and Profiling

[![Author](https://img.shields.io/badge/Author-Prakash%20Ukhalkar-blue.svg)](https://github.com/prakash-ukhalkar) [![GitHub Repository](https://img.shields.io/badge/GitHub-Repo-lightgrey)](https://github.com/prakash-ukhalkar/Advanced-Credit-Risk-Prediction) [![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![Pandas](https://img.shields.io/badge/Pandas-Latest-green)](https://pandas.pydata.org/) [![NumPy](https://img.shields.io/badge/NumPy-Latest-lightblue)](https://numpy.org/) [![statsmodels](https://img.shields.io/badge/statsmodels-Latest-orange)](https://www.statsmodels.org/) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

**Objective:** Establish a "Dual-Track" data ingestion pipeline to independently load and profile Individual/Retail datasets versus Corporate datasets.

**Introduction:** To effectively model credit risk across vastly different economic entities, this notebook strictly adheres to a "Dual-Track" methodology during initial ingestion. This isolates Retail (Individual) data from Corporate data. The codebase systematically reads the raw data files, handles potential ingestion errors gracefully, and performs an initial statistical profiling on both tracks to inform subsequent exploratory data analysis (EDA).

## Step 1: Environment Setup
Importing core data manipulation libraries and suppressing warnings for a clean aesthetic. Defining the relative path to the raw data repository.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

RAW_DATA_DIR = os.path.join('..', 'data', 'raw')

## Step 2: Retail Data Ingestion
The Retail track consists of individual-level demographic information, historical loan performances, and external metric evaluations. We programmatically load these specific CSV/Excel exports into the `retail_data_dict` dictionary.

In [4]:
retail_files = [
    'Credit Risk_Individual_Kaggel_26022026.csv',
    'loan_data_of-indiiduals_demography_creditscore_26022026.csv',
    'default_credit card clients.xls',
    'External_Cibil_Dataset.xlsx'
]

retail_data_dict = {}
print("Initiating Retail Data Ingestion Pipeline...")

for file_name in retail_files:
    file_path = os.path.join(RAW_DATA_DIR, file_name)
    try:
        # Check file extension and load accordingly
        if file_name.endswith('.csv'):
            df = pd.read_csv(file_path)
        elif file_name.endswith('.xls') or file_name.endswith('.xlsx'):
            df = pd.read_excel(file_path)
        else:
            print(f"[WARNING] Unrecognized format for '{file_name}'")
            continue
            
        retail_data_dict[file_name] = df
        print(f"[SUCCESS] Loaded '{file_name}' | Shape: {df.shape}")
    except FileNotFoundError:
        print(f"[ERROR] FileNotFoundError: Could not locate '{file_path}'")
    except Exception as e:
        print(f"[ERROR] I/O exception '{file_name}': {e}")

Initiating Retail Data Ingestion Pipeline...
[SUCCESS] Loaded 'Credit Risk_Individual_Kaggel_26022026.csv' | Shape: (1000, 21)
[SUCCESS] Loaded 'loan_data_of-indiiduals_demography_creditscore_26022026.csv' | Shape: (45000, 14)
[SUCCESS] Loaded 'default_credit card clients.xls' | Shape: (30001, 25)
[SUCCESS] Loaded 'External_Cibil_Dataset.xlsx' | Shape: (51336, 62)


## Step 3: Corporate Data Ingestion
The Corporate track covers datasets detailing corporate defaults, systematic financial risk features, and broader solvency assessments. Analogous to the retail sequence, these are aggregated into the `corporate_data_dict` dictionary.

In [5]:
corporate_files = [
    'Corporate Default Financial Dataset.csv',
    'Corporate_Financial_Risk_Assessment_Data_26022026.csv'
]

corporate_data_dict = {}
print("\nInitiating Corporate Data Ingestion Pipeline...")

for file_name in corporate_files:
    file_path = os.path.join(RAW_DATA_DIR, file_name)
    try:
        df = pd.read_csv(file_path)
        corporate_data_dict[file_name] = df
        print(f"[SUCCESS] Loaded '{file_name}' | Shape: {df.shape}")
    except FileNotFoundError:
        print(f"[ERROR] FileNotFoundError: Could not locate '{file_path}'")
    except Exception as e:
        print(f"[ERROR] I/O exception '{file_name}': {e}")


Initiating Corporate Data Ingestion Pipeline...
[SUCCESS] Loaded 'Corporate Default Financial Dataset.csv' | Shape: (92048, 24)
[SUCCESS] Loaded 'Corporate_Financial_Risk_Assessment_Data_26022026.csv' | Shape: (5000, 21)


## Step 4: Statistical Profiling
To reliably establish foundational data quality assumptions, we employ a systematic profiling loop. The `profile_datasets` function iterates over the populated data dictionaries. It surfaces deterministic statistics for each DataFrame.

In [6]:
def profile_datasets(data_dict: dict, track_name: str) -> None:
    if not data_dict:
        print(f"Skipping profiling: The '{track_name}' data dictionary is empty.\n")
        return
        
    print("=" * 70)
    print(f" DATA PROFILING REPORT: {track_name.upper()} TRACK ")
    print("=" * 70 + "\n")
    
    for name, df in data_dict.items():
        print(f"=== {name} ===")
        print(f"Shape: {df.shape}")
        missing_pct = (df.isnull().sum() / len(df)) * 100
        print(f"Missing Values Overall: {missing_pct.mean().round(2)}%")
        print("-" * 30 + "\n")

print("\n--- DATA PROFILING ---")
profile_datasets(retail_data_dict, track_name="Retail")
profile_datasets(corporate_data_dict, track_name="Corporate")


--- DATA PROFILING ---
 DATA PROFILING REPORT: RETAIL TRACK 

=== Credit Risk_Individual_Kaggel_26022026.csv ===
Shape: (1000, 21)
Missing Values Overall: 0.0%
------------------------------

=== loan_data_of-indiiduals_demography_creditscore_26022026.csv ===
Shape: (45000, 14)
Missing Values Overall: 0.0%
------------------------------

=== default_credit card clients.xls ===
Shape: (30001, 25)
Missing Values Overall: 0.0%
------------------------------

=== External_Cibil_Dataset.xlsx ===
Shape: (51336, 62)
Missing Values Overall: 0.0%
------------------------------

 DATA PROFILING REPORT: CORPORATE TRACK 

=== Corporate Default Financial Dataset.csv ===
Shape: (92048, 24)
Missing Values Overall: 12.15%
------------------------------

=== Corporate_Financial_Risk_Assessment_Data_26022026.csv ===
Shape: (5000, 21)
Missing Values Overall: 4.76%
------------------------------



### Key Findings
Based on the initial data ingestion and profiling process:
- **Data Sparsity Highlights:** The Retail Track data is well-populated with an average of 0.0% missing values across all four individual datasets. Conversely, the Corporate Track exhibits noticeable sparsity; the *Corporate Default Financial Dataset* contains ~12.15% missing data overall, and the *Financial Risk Assessment* data shows ~4.76% missing fields. This structural sparsity in corporate features will require dedicated imputation strategies in Notebook 02.
- **Dimensionality Observations:** The Retail dataset volumes range from 1,000 to over 51,000 observations per file, supplying rich individual-level behavior patterns. The Corporate tracking is primarily driven by the robust 92,048 records in the Default Financial Dataset.
- **Pipeline Integrity:** The "Dual-Track" separation successfully partitioned the distinct structures of retail metrics from corporate financial indicators, confirming that early cross-domain table merging would have resulted in severe feature misalignment.

---
### End of Notebook 01 — Data Acquisition and Profiling

**Outputs produced:**
- `None` — Structural data loading into memory via DataFrames only.

**Next step → Notebook 02:** Exploratory Data Analysis & Feature Engineering. Initial univariate/bivariate visualizations and initiate temporal modeling functions.

<div align="center"><sub>END OF NOTEBOOK 01</sub></div>